# Screen recorder

Records a selected monitor to MP4 until you stop it. A JSON sidecar is written next to the video with source monitor, timestamps, and host info.

**Workflow:** run the setup cell once, list monitors, set `MONITOR`, call `start_recording()`, then `stop_recording()` when done.

In [1]:
# Run once per kernel — installs capture dependencies
%pip install -q -r scripts/screen_record-requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.screen_recorder import ScreenRecorder, print_monitors

OUTPUT_DIR = ROOT / "recordings"
FPS = 30

recorder: ScreenRecorder | None = None

In [2]:
# Pick a monitor number from the list below (1, 2, 3, ...)
print_monitors()

Monitor 1: 2560x1440 @ (-2560, 0)
Monitor 2: 2560x1440 @ (0, 0)


[MonitorInfo(index=1, left=-2560, top=0, width=2560, height=1440),
 MonitorInfo(index=2, left=0, top=0, width=2560, height=1440)]

In [3]:
MONITOR = 1  # change to 2, 3, ... based on the list above


def start_recording(monitor: int = MONITOR) -> Path:
    global recorder
    if recorder is not None and recorder.is_recording:
        raise RuntimeError("Already recording. Run stop_recording() first.")

    recorder = ScreenRecorder(
        monitor=monitor,
        output_dir=OUTPUT_DIR,
        fps=FPS,
    )
    video_path = recorder.start()
    print(f"Recording monitor {monitor} -> {video_path}")
    return video_path


start_recording()

Recording monitor 1 -> d:\repos\GuideAnts\recordings\screen_m1_20260708_181821.mp4


WindowsPath('d:/repos/GuideAnts/recordings/screen_m1_20260708_181821.mp4')

In [4]:
def stop_recording() -> dict:
    global recorder
    if recorder is None:
        raise RuntimeError("No recorder. Run start_recording() first.")
    if not recorder.is_recording:
        raise RuntimeError("Not currently recording.")

    metadata = recorder.stop()
    print(f"Saved video: {recorder.video_path}")
    print(f"Saved metadata: {recorder.metadata_path}")
    return metadata


metadata = stop_recording()
metadata

Saved video: d:\repos\GuideAnts\recordings\screen_m1_20260708_181821.mp4
Saved metadata: d:\repos\GuideAnts\recordings\screen_m1_20260708_181821.json


{'recording': {'started_at': '2026-07-08T18:18:21.206-04:00',
  'stopped_at': '2026-07-08T18:18:49.133-04:00',
  'duration_seconds': 27.927},
 'source': {'type': 'monitor',
  'monitor_index': 1,
  'left': -2560,
  'top': 0,
  'width': 2560,
  'height': 1440},
 'host': {'hostname': 'OFFICEDESKTOP',
  'platform': 'Windows-11-10.0.26200-SP0'},
 'video': {'path': 'D:\\repos\\GuideAnts\\recordings\\screen_m1_20260708_181821.mp4',
  'fps': 30,
  'frame_count': 801,
  'codec': 'mp4v',
  'container': 'mp4'}}